In [ ]:
# ================================================================
# GENERAL 1D TRANSIENT HEAT CONDUCTION USING EXPLICIT FDM (FTCS)
# ================================================================
#
# Governing equation:
#
#              dT/dt = alpha d²T/dx²
#
#
# Boundary conditions:
#
#              T(0,t) = T_left
#              T(L,t) = T_right
#
#
# Initial condition:
#
#              T(x,0) = T_initial
#
#
# Assumptions:
#
#   1. One-dimensional transient heat conduction
#   2. Constant thermal properties
#   3. No internal heat generation
#   4. Prescribed temperatures at both boundaries
#
# ================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ================================================================
# 1. USER INPUTS
# ================================================================
#
# CHANGE THESE VALUES FOR DIFFERENT TRANSIENT PROBLEMS
# ================================================================

L = 0.4                 # Wall length (m)

N = 5                   # Total number of spatial grid nodes

alpha = 1.0e-4          # Thermal diffusivity (m²/s)


# Initial temperature of the interior

T_initial = 20.0        # °C


# Boundary temperatures for t > 0

T_left = 100.0          # °C

T_right = 100.0         # °C


# Time discretization

dt = 25.0               # Time step (s)

t_final = 500.0         # Final simulation time (s)


# Number of Fourier terms for analytical solution

analytical_terms = 100


# ================================================================
# 2. BASIC INPUT CHECKS
# ================================================================

if N < 3:
    raise ValueError("N must be at least 3.")

if L <= 0:
    raise ValueError("L must be positive.")

if alpha <= 0:
    raise ValueError("Thermal diffusivity alpha must be positive.")

if dt <= 0:
    raise ValueError("Time step dt must be positive.")

if t_final <= 0:
    raise ValueError("Final time must be positive.")


# ================================================================
# 3. SPATIAL GRID
# ================================================================

dx = L / (N - 1)

x = np.linspace(0, L, N)


print("================================================")
print("SPATIAL GRID INFORMATION")
print("================================================")

print("Wall length L            =", L, "m")
print("Number of nodes N        =", N)
print("Grid spacing dx          =", dx, "m")
print("Node positions           =", x)

print()


# ================================================================
# 4. FOURIER NUMBER
# ================================================================
#
#              Fo = alpha dt / dx²
#
# For the explicit FTCS method:
#
#              Fo <= 0.5
#
# is required for stability in 1D.
#
# ================================================================

Fo = alpha * dt / dx**2


# Maximum stable time step

dt_max = dx**2 / (2.0 * alpha)


print("================================================")
print("STABILITY INFORMATION")
print("================================================")

print("Thermal diffusivity alpha =", alpha, "m²/s")
print("Time step dt              =", dt, "s")
print("Fourier number Fo         =", Fo)
print("Maximum stable dt         =", dt_max, "s")

print()


if Fo <= 0.5:

    print("STABILITY CHECK: STABLE")
    print("Fo <= 0.5")

else:

    print("STABILITY CHECK: UNSTABLE")
    print("Fo > 0.5")

    raise ValueError(
        "The explicit FTCS scheme is unstable. "
        "Reduce dt or increase dx."
    )


print()


# ================================================================
# 5. TIME GRID
# ================================================================
#
# Number of time steps required to reach t_final
#
# ================================================================

num_time_steps = int(round(t_final / dt))


# Make sure t_final is compatible with dt

if not np.isclose(num_time_steps * dt, t_final):

    raise ValueError(
        "t_final must be an integer multiple of dt "
        "for this teaching version of the code."
    )


time = np.arange(
    num_time_steps + 1
) * dt


print("================================================")
print("TIME GRID INFORMATION")
print("================================================")

print("Final time               =", t_final, "s")
print("Number of time steps     =", num_time_steps)
print("Time levels              =", time)

print()


# ================================================================
# 6. INITIAL TEMPERATURE FIELD
# ================================================================
#
# At t = 0:
#
# Interior:
#
#              T = T_initial
#
# Boundaries:
#
#              T_left and T_right
#
# The boundary temperatures are taken to change immediately
# to their prescribed values at t = 0.
#
# ================================================================

T = np.full(
    N,
    T_initial,
    dtype=float
)


# Apply boundary temperatures

T[0] = T_left

T[-1] = T_right


print("================================================")
print("INITIAL TEMPERATURE VECTOR")
print("================================================")

print(T)

print()


# ================================================================
# 7. STORAGE FOR COMPLETE TIME HISTORY
# ================================================================
#
# Rows    -> time levels
# Columns -> spatial nodes
#
# ================================================================

T_history = np.zeros(
    (num_time_steps + 1, N)
)


# Store initial field

T_history[0, :] = T


# ================================================================
# 8. EXPLICIT FTCS FINITE DIFFERENCE FORMULA
# ================================================================
#
# Governing equation:
#
#              dT/dt = alpha d²T/dx²
#
#
# Forward difference in time:
#
#       T_i^(n+1) - T_i^n
#       -----------------
#               dt
#
#
# Central difference in space:
#
#       T_(i-1)^n - 2T_i^n + T_(i+1)^n
#       --------------------------------
#                     dx²
#
#
# Therefore:
#
#       T_i^(n+1)
#
#       = T_i^n
#
#       + Fo [T_(i-1)^n - 2T_i^n + T_(i+1)^n]
#
#
# or:
#
#       T_i^(n+1)
#
#       = Fo T_(i-1)^n
#
#       + (1 - 2Fo) T_i^n
#
#       + Fo T_(i+1)^n
#
# ================================================================


print("================================================")
print("EXPLICIT FTCS UPDATE EQUATION")
print("================================================")

print(
    "T_i^(n+1) = Fo*T_(i-1)^n "
    "+ (1-2Fo)*T_i^n "
    "+ Fo*T_(i+1)^n"
)

print()

print(
    f"For this problem: "
    f"T_i^(n+1) = "
    f"{Fo:.4f}*T_(i-1)^n "
    f"+ {1 - 2*Fo:.4f}*T_i^n "
    f"+ {Fo:.4f}*T_(i+1)^n"
)

print()


# ================================================================
# 9. TIME-MARCHING LOOP
# ================================================================
#
# IMPORTANT:
#
# Unlike the steady-state examples, we do NOT solve A*T = b.
#
# Instead:
#
#       T^0 -> T^1 -> T^2 -> T^3 -> ...
#
# Each new time level is calculated from the previous time level.
#
# ================================================================

for n in range(num_time_steps):


    # ------------------------------------------------------------
    # Create a copy for the new time level
    # ------------------------------------------------------------

    T_new = T.copy()


    # ------------------------------------------------------------
    # Update only the INTERIOR nodes
    # ------------------------------------------------------------

    for i in range(1, N - 1):


        T_new[i] = (

            Fo * T[i - 1]

            + (1.0 - 2.0 * Fo) * T[i]

            + Fo * T[i + 1]

        )


    # ------------------------------------------------------------
    # Re-apply fixed boundary temperatures
    # ------------------------------------------------------------

    T_new[0] = T_left

    T_new[-1] = T_right


    # ------------------------------------------------------------
    # Move to next time level
    # ------------------------------------------------------------

    T = T_new.copy()


    # Store result

    T_history[n + 1, :] = T


# ================================================================
# 10. DISPLAY COMPLETE TIME-MARCHING TABLE
# ================================================================

table_data = {

    "Time (s)": time
}


for i in range(N):

    table_data[f"T{i} (°C)"] = np.round(
        T_history[:, i],
        5
    )


time_table = pd.DataFrame(
    table_data
)


print("================================================")
print("TIME-MARCHING FDM RESULTS")
print("================================================")

display(time_table)


# ================================================================
# 11. ANALYTICAL SOLUTION
# ================================================================
#
# The analytical solution used here is valid for:
#
#   1. Uniform initial temperature T_initial
#   2. Equal fixed boundary temperatures:
#
#              T_left = T_right = T_boundary
#
#
# General form:
#
# T(x,t)
#
# = T_boundary
#
# + 4(T_initial - T_boundary)/pi
#
#   * Sum over odd n:
#
#       [1/n]
#
#       sin(n pi x/L)
#
#       exp[-alpha(n pi/L)² t]
#
# ================================================================


analytical_available = np.isclose(
    T_left,
    T_right
)


def analytical_temperature(
    x_values,
    t_value,
    terms=100
):


    if not analytical_available:

        return None


    T_boundary = T_left


    T_exact = np.full_like(
        x_values,
        T_boundary,
        dtype=float
    )


    for m in range(terms):


        # Odd terms:
        #
        # n = 1, 3, 5, ...

        n = 2 * m + 1


        T_exact += (

            4.0
            * (T_initial - T_boundary)
            / (n * np.pi)

            * np.sin(
                n * np.pi * x_values / L
            )

            * np.exp(

                -alpha

                * (
                    n * np.pi / L
                )**2

                * t_value

            )

        )


    return T_exact


# ================================================================
# 12. ANALYTICAL COMPARISON AT FINAL TIME
# ================================================================

if analytical_available:


    T_exact_final = analytical_temperature(
        x,
        t_final,
        analytical_terms
    )


    abs_error = np.abs(
        T_history[-1, :]
        - T_exact_final
    )


    comparison = pd.DataFrame({

        "Node i":
            np.arange(N),

        "x (m)":
            np.round(x, 5),

        "FDM T (°C)":
            np.round(
                T_history[-1, :],
                6
            ),

        "Exact T (°C)":
            np.round(
                T_exact_final,
                6
            ),

        "Absolute Error (°C)":
            np.round(
                abs_error,
                6
            )

    })


    print("================================================")
    print(
        f"FDM VS ANALYTICAL SOLUTION AT t = {t_final} s"
    )
    print("================================================")

    display(comparison)


else:

    print("================================================")
    print("ANALYTICAL COMPARISON NOT USED")
    print("================================================")

    print(
        "This particular Fourier-series formula assumes "
        "equal prescribed temperatures at both boundaries."
    )

    print()


# ================================================================
# 13. PRINT FINAL TEMPERATURE VECTOR
# ================================================================

print("================================================")
print("FINAL FDM TEMPERATURE VECTOR")
print("================================================")

print(
    f"At t = {t_final} s:"
)

print(
    np.round(
        T_history[-1, :],
        6
    )
)

print()


# ================================================================
# 14. SELECT TIME LEVELS TO DISPLAY ON THE GRAPH
# ================================================================
#
# For a small teaching problem, plotting every time level is fine.
#
# For a long simulation with hundreds of time levels, displaying
# all curves would make the graph difficult to read.
#
# Therefore, choose a few representative times automatically.
#
# ================================================================

if num_time_steps <= 8:

    plot_indices = np.arange(
        num_time_steps + 1
    )

else:

    plot_indices = np.unique(
        np.linspace(
            0,
            num_time_steps,
            6,
            dtype=int
        )
    )


# ================================================================
# 15. FDM TIME-EVOLUTION PLOT
# ================================================================
#
# FDM values are discrete nodal values.
#
# Therefore they are shown as POINTS.
#
# ================================================================

plt.figure(
    figsize=(9, 6)
)


for n in plot_indices:

    plt.scatter(

        x,

        T_history[n, :],

        s=55,

        label=f"t = {time[n]:.0f} s"

    )


plt.xlabel(
    "Position x (m)"
)

plt.ylabel(
    "Temperature T (°C)"
)

plt.title(
    "Transient 1D Heat Conduction\n"
    "Explicit FDM Time Evolution"
)

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()


# ================================================================
# 16. FDM + ANALYTICAL COMPARISON AT FINAL TIME
# ================================================================

if analytical_available:


    # Smooth spatial grid for analytical solution

    x_dense = np.linspace(
        0,
        L,
        300
    )


    T_exact_dense = analytical_temperature(
        x_dense,
        t_final,
        analytical_terms
    )


    plt.figure(
        figsize=(9, 5)
    )


    # Analytical solution:
    # continuous line

    plt.plot(

        x_dense,

        T_exact_dense,

        linewidth=2,

        label="Analytical Solution"

    )


    # FDM solution:
    # discrete nodal points

    plt.scatter(

        x,

        T_history[-1, :],

        s=80,

        c="darkred",

        label=f"FDM at t = {t_final:.0f} s"

    )


    plt.xlabel(
        "Position x (m)"
    )

    plt.ylabel(
        "Temperature T (°C)"
    )

    plt.title(
        f"FDM vs Analytical Solution at t = {t_final:.0f} s"
    )

    plt.grid(True)

    plt.legend()

    plt.tight_layout()

    plt.show()










    # ================================================================
# SPACE-TIME TEMPERATURE CONTOUR
# ================================================================
#
# Horizontal axis : position x
# Vertical axis   : time t
# Color           : temperature T(x,t)
#
# ================================================================

X_time, Time_mesh = np.meshgrid(x, time)

plt.figure(figsize=(9, 6))

contour = plt.contourf(
    X_time,
    Time_mesh,
    T_history,
    levels=20
)

plt.colorbar(
    contour,
    label="Temperature T (°C)"
)

plt.xlabel("Position x (m)")

plt.ylabel("Time t (s)")

plt.title(
    "Space-Time Temperature Contour\n"
    "1D Transient Heat Conduction"
)

plt.tight_layout()

plt.show()


# ================================================================
# 17. CENTER-NODE TEMPERATURE VERSUS TIME
# ================================================================
#
# This graph is particularly useful for understanding
# the meaning of transient heat conduction.
#
# ================================================================

center_index = N // 2


plt.figure(
    figsize=(8, 5)
)


plt.plot(

    time,

    T_history[:, center_index],

    marker="o"

)


plt.xlabel(
    "Time t (s)"
)

plt.ylabel(
    f"Temperature at node {center_index} (°C)"
)

plt.title(
    "Temperature Evolution at the Center of the Wall"
)

plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
import plotly.graph_objects as go

# Create a 3D Surface Plot of the Temperature Profile
fig = go.Figure(data=[go.Surface(z=T_history, x=x, y=time, colorscale='Viridis')])

fig.update_layout(
    title='Spatiotemporal Temperature Evolution',
    scene=dict(
        xaxis_title='Position x (m)',
        yaxis_title='Time t (s)',
        zaxis_title='Temperature T (°C)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    autosize=False,
    width=800,
    height=700,
    margin=dict(l=65, r=50, b=65, t=90)
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Prepare frames for animation
frames = []
for n in range(len(time)):
    frames.append(go.Frame(
        data=[go.Scatter(x=x, y=T_history[n, :], mode='lines+markers',
                         marker=dict(color='red', size=10),
                         line=dict(color='blue', width=3))],
        name=str(time[n])
    ))

# Create initial figure
fig = go.Figure(
    data=[go.Scatter(x=x, y=T_history[0, :], mode='lines+markers',
                     marker=dict(color='red', size=10),
                     line=dict(color='blue', width=3))],
    layout=go.Layout(
        title="Interactive Thermal Evolution Animation",
        xaxis=dict(range=[0, L], title="Position x (m)"),
        yaxis=dict(range=[T_initial-5, T_left+5], title="Temperature T (°C)"),
        updatemenus=[dict(type="buttons",
                          buttons=[dict(label="Play",
                                        method="animate",
                                        args=[None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}]),
                                   dict(label="Pause",
                                        method="animate",
                                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}])])]
    ),
    frames=frames
)

# Add slider
fig.update_layout(
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {"font": {"size": 20}, "prefix": "Time: ", "visible": True, "xanchor": "right"},
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "y": 0,
        "steps": [{"args": [[f.name], {"frame": {"duration": 300, "redraw": True}, "mode": "immediate", "transition": {"duration": 300}}],
                   "label": f.name, "method": "animate"} for f in frames]
    }]
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# Create a dynamic heatmap to visualize heat penetration over time
fig = go.Figure(data=go.Heatmap(
    z=T_history,
    x=x,
    y=time,
    colorscale='Hot',
    colorbar=dict(title='Temp (°C)')
))

fig.update_layout(
    title='Thermal Intensity Map (Time vs. Position)',
    xaxis_title='Position x (m)',
    yaxis_title='Time t (s)',
    width=800,
    height=500
)

fig.show()

In [ ]:
# 1. Increase spatial resolution (more nodes)
N = 50
dx = L / (N - 1)
x = np.linspace(0, L, N)

# 2. Decrease time step to ensure high accuracy and stability
# (dt must be <= dx^2 / (2*alpha))
dt = 0.5 * (dx**2 / (2.0 * alpha))
num_time_steps = int(t_final / dt)
time = np.arange(num_time_steps + 1) * dt
Fo = alpha * dt / dx**2

# 3. Re-run simulation
T = np.full(N, T_initial, dtype=float)
T[0], T[-1] = T_left, T_right
T_history = np.zeros((num_time_steps + 1, N))
T_history[0, :] = T

for n in range(num_time_steps):
    T_new = T.copy()
    T_new[1:-1] = Fo * T[:-2] + (1.0 - 2.0 * Fo) * T[1:-1] + Fo * T[2:]
    T = T_new
    T_history[n + 1, :] = T

# 4. Visualize the improved accuracy at final time
x_dense = np.linspace(0, L, 500)
T_exact = analytical_temperature(x_dense, t_final)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(x_dense, T_exact, 'k-', label='Exact Analytical', linewidth=2)
plt.scatter(x, T_history[-1, :], color='cyan', s=20, label=f'Accurate FDM (N={N})', alpha=0.6)
plt.title(f'Improved Accuracy: FDM vs Analytical (t={t_final}s)')
plt.xlabel('Position x (m)')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"New Fourier Number: {Fo:.4f}")
print(f"Nodes: {N}, Time Steps: {num_time_steps}")